## 实现线性回归数据

|   任务    |   方法    |   日期    |
| ---       |       --- | ---       |
|softmax分类|pytorch    |2026.11.1  |

In [2]:
%matplotlib inline
import random
import torch 
from d2l import torch as d2l

In [3]:
def set_figsize(figsize=(3.5, 2.5)):
    """Set the figure size for matplotlib."""
    d2l.plt.rcParams['figure.figsize'] = figsize

In [4]:
def set_axes(axes, xlabel, ylabel, xlim, ylim, xscale, yscale, legend):
    """Set the axes for matplotlib."""
    axes.set_xlabel(xlabel)
    axes.set_ylabel(ylabel)
    axes.set_xscale(xscale)
    axes.set_yscale(yscale)
    axes.set_xlim(xlim)
    axes.set_ylim(ylim)
    if legend:
        axes.legend(legend)
    axes.grid()

In [5]:
def synthetic_data(w, b, num_examples):  #@save
    """Generate y = Xw + b + noise."""
    X = torch.normal(0, 1, (num_examples, len(w)))
    y = torch.matmul(X, w) + b
    y += torch.normal(0, 0.01, y.shape)
    return X, y.reshape((-1, 1))

In [6]:
true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = synthetic_data(true_w, true_b, 1000)
print('features:', features[0],'\nlabel:', labels[0])

features: tensor([ 0.5293, -0.2588]) 
label: tensor([6.1348])


In [7]:
def data_iter(batch_size, features, labels):
    """Generate a minibatch of data."""
    num_examples = len(features)
    indices = list(range(num_examples))
    random.shuffle(indices)  # Shuffle the indices
    for i in range(0, num_examples, batch_size):
        batch_indices = torch.tensor(
            indices[i: min(i + batch_size, num_examples)])
        yield features[batch_indices], labels[batch_indices]

### 知识点：**生成器**
>当代码执行到 yield 的时候，函数会立刻“暂停”并冻结在这里，把当前这批数据“吐”出来交给你。
>
>此时，函数内部的 for 循环就停在这一行，不再往下走了。
>
>同时，它会记住自己现在的状态（比如记住现在 i 走到了多少，哪些索引已经被打乱了）。

In [8]:
batch_size = 10
for X, y in data_iter(batch_size, features, labels):
    print(X, '\n', y)
    break
    

tensor([[-0.1901, -1.9460],
        [-0.9840,  0.4430],
        [ 1.8871, -1.4991],
        [ 0.3629,  0.4451],
        [ 0.3170,  0.2700],
        [ 0.4519,  0.8734],
        [ 1.0124, -1.0010],
        [-0.4902,  0.3076],
        [ 0.8215,  0.9976],
        [-0.7144, -0.1713]]) 
 tensor([[10.4406],
        [ 0.7189],
        [13.0474],
        [ 3.4266],
        [ 3.9071],
        [ 2.1344],
        [ 9.6181],
        [ 2.1657],
        [ 2.4454],
        [ 3.3583]])


In [9]:
w = torch.normal(0, 0.01, size=(2, 1), requires_grad=True)
b = torch.zeros(1, requires_grad=True)

In [10]:
def squared_loss(y_hat, y):  #@save
    """Mean squared error loss."""
    return (y_hat - y.reshape(y_hat.shape)) ** 2 / 2

In [11]:
def sgd(params, lr, batch_size):  #@save
    """
    优化器

    参数：
        params: 
            待优化的参数列表
        lr: 
            学习率
        batch_size: 
            批次大小
    """
    with torch.no_grad():
        for param in params:
            param -= lr * param.grad / batch_size
            param.grad.zero_()

In [12]:
def linreg(X, w, b):  #@save
    """Linear regression model."""
    return torch.matmul(X, w) + b

### 超参数定义

In [14]:
epochs = 3
lr = 0.03
net = linreg
loss = squared_loss

In [15]:
for epoch in range(epochs):
    for X, y in data_iter(batch_size, features, labels):
        l = loss(net(X, w, b), y)  # Compute loss
        l.sum().backward()  # Backpropagate
        sgd([w, b], lr, batch_size)  # Update parameters
    with torch.no_grad():
        train_l = loss(net(features, w, b), labels)
        print(f'epoch {epoch + 1}, loss {float(train_l.mean()):f}')

epoch 1, loss 0.045552
epoch 2, loss 0.000202
epoch 3, loss 0.000048


In [16]:
print(f'w的估计误差: {true_w - w.reshape(true_w.shape)}')
print(f'b的估计误差: {true_b - b}')

w的估计误差: tensor([-0.0001, -0.0009], grad_fn=<SubBackward0>)
b的估计误差: tensor([0.0004], grad_fn=<RsubBackward1>)
